In [1]:
import geemap
import ee
geemap.ee_initialize()
ee.Authenticate()

Enter verification code:  [REDACTED]



Successfully saved authorization token.


True

In [3]:
import ee
import geemap

# -----------------------------
# 1) Initialize
# -----------------------------
try:
    ee.Initialize()
except Exception:
    geemap.ee_initialize()
    ee.Initialize()

# -----------------------------
# 2) ROI (Haor bbox)
# -----------------------------
BBOX_BD = [89.8, 23.5, 92.8, 25.5]
roi = ee.Geometry.Rectangle(BBOX_BD)

# -----------------------------
# 3) Dates (edit)
# -----------------------------
REF_START   = "2026-01-01"
REF_END     = "2026-01-30"
FLOOD_START = "2025-06-01"
FLOOD_END   = "2025-06-30"

# Water index thresholds (tune)
MNDWI_WATER_THR = 0.10        # flood water condition (MNDWI > this)
DELTA_MNDWI_THR = 0.15        # flood-change condition (flood-ref > this)

# -----------------------------
# 4) Helpers
# -----------------------------
def count_ic(ic, label):
    n = ic.size().getInfo()
    print(f"[DIAG] {label}: {n}")
    return n

def safe_median_singleband(ic, band_name):
    empty = ee.Image.constant(0).rename(band_name).updateMask(ee.Image.constant(0))
    return ee.Image(ee.Algorithms.If(ic.size().gt(0),
                                     ic.median().rename(band_name),
                                     empty))

def safe_median_multiband(ic, band_names):
    # returns empty masked image with those bands if ic is empty
    empty = ee.Image.constant([0]*len(band_names)).rename(band_names).updateMask(ee.Image.constant(0))
    return ee.Image(ee.Algorithms.If(ic.size().gt(0),
                                     ic.median().select(band_names),
                                     empty))

def mask_landsat_sr(img):
    """
    Landsat Collection 2 Level-2 (SR) QA mask using QA_PIXEL.
    Masks: fill, cloud, cloud shadow, cirrus, snow.
    """
    qa = img.select("QA_PIXEL")
    # bits: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
    # 0 Fill, 1 Dilated Cloud, 2 Cirrus, 3 Cloud, 4 Cloud Shadow, 5 Snow
    mask = (qa.bitwiseAnd(1 << 0).eq(0)
            .And(qa.bitwiseAnd(1 << 1).eq(0))
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0)))
    return img.updateMask(mask)

def prep_landsat():
    # Landsat 8 + 9 L2 SR
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    col = l8.merge(l9).filterBounds(roi)

    # Cloud mask
    col = col.map(mask_landsat_sr)

    # Convert SR bands to reflectance (scale)
    # SR_B* scale=0.0000275, offset=-0.2 (per dataset docs)
    def scale_sr(img):
        optical = img.select(["SR_B2","SR_B3","SR_B4","SR_B5","SR_B6","SR_B7"]) \
                     .multiply(0.0000275).add(-0.2)
        return img.addBands(optical, overwrite=True)

    col = col.map(scale_sr)

    # Add MNDWI = (Green - SWIR1) / (Green + SWIR1)
    # Green=SR_B3, SWIR1=SR_B6
    def add_mndwi(img):
        mndwi = img.normalizedDifference(["SR_B3","SR_B6"]).rename("MNDWI")
        return img.addBands(mndwi)

    col = col.map(add_mndwi)

    return col

# -----------------------------
# 5) Build collections
# -----------------------------
ls = prep_landsat()

ref_col   = ls.filterDate(REF_START, REF_END)
flood_col = ls.filterDate(FLOOD_START, FLOOD_END)

ref_n = count_ic(ref_col,   f"Landsat REF ({REF_START} to {REF_END})")
flood_n = count_ic(flood_col, f"Landsat FLOOD ({FLOOD_START} to {FLOOD_END})")

if ref_n == 0 or flood_n == 0:
    print("[WARNING] Landsat has 0 usable scenes in one window (clouds are common in monsoon).")
    print("          If this happens, consider Sentinel-2 or widen the date window.")

# Composites
# RGB: Red=SR_B4, Green=SR_B3, Blue=SR_B2
ref_rgb   = safe_median_multiband(ref_col,   ["SR_B4","SR_B3","SR_B2"]).clip(roi)
flood_rgb = safe_median_multiband(flood_col, ["SR_B4","SR_B3","SR_B2"]).clip(roi)

ref_mndwi   = safe_median_singleband(ref_col.select("MNDWI"),   "ref_mndwi").clip(roi)
flood_mndwi = safe_median_singleband(flood_col.select("MNDWI"), "flood_mndwi").clip(roi)

delta_mndwi = flood_mndwi.subtract(ref_mndwi).rename("delta_mndwi")

# Flood mask (two ways; you can use either or AND/OR them)
# A) Change-based
mask_change = delta_mndwi.gt(DELTA_MNDWI_THR)

# B) Water-in-flood but not water-in-ref
mask_water_new = flood_mndwi.gt(MNDWI_WATER_THR).And(ref_mndwi.lte(MNDWI_WATER_THR))

# Combined flood mask (more strict)
flood_mask = mask_change.And(mask_water_new).rename("flood_mask")

# -----------------------------
# 6) Visualize
# -----------------------------
m = geemap.Map()
m.centerObject(roi, 7)

m.addLayer(ref_rgb,   {"min": 0.0, "max": 0.3}, "Landsat REF RGB", False)
m.addLayer(flood_rgb, {"min": 0.0, "max": 0.3}, "Landsat FLOOD RGB", True)

m.addLayer(ref_mndwi,   {"min": -0.5, "max": 0.5}, "REF MNDWI", False)
m.addLayer(flood_mndwi, {"min": -0.5, "max": 0.5}, "FLOOD MNDWI", False)

m.addLayer(delta_mndwi, {"min": -0.3, "max": 0.6}, "ΔMNDWI (flood-ref)", True)
m.addLayer(flood_mask.selfMask(), {"palette": ["FF0000"]}, "Flood mask (Landsat)", True)

m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "ROI", True)

m


[DIAG] Landsat REF (2026-01-01 to 2026-01-30): 44
[DIAG] Landsat FLOOD (2025-06-01 to 2025-06-30): 23


Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…